# Reliance Stock Price Prediction

**Machine Learning Minor Project**

This project uses historical Reliance Industries stock data to estimate the next trading day's closing price using machine learning.

## 1. Install and Import Libraries

In [ ]:
!pip -q install yfinance gradio

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

## 2. Load Historical Stock Data

In [ ]:
data = yf.download(
    "RELIANCE.NS",
    start="2015-01-01",
    end="2026-01-01"
)

# yfinance may return multi-level columns; keep the normal column names.
if isinstance(data.columns, pd.MultiIndex):
    data.columns = data.columns.get_level_values(0)

data.head()

## 3. Understand the Dataset

In [ ]:
print("Number of rows and columns:", data.shape)
print("\nDataset information:")
data.info()

In [ ]:
print("Summary statistics:")
data.describe()

In [ ]:
print("Missing values:")
print(data.isnull().sum())

print("\nDuplicate rows:", data.duplicated().sum())

In [ ]:
print("Columns:")
print(data.columns)

## 4. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(data.index, data["Close"], label="Closing Price")

plt.title("Reliance Industries Closing Price")
plt.xlabel("Date")
plt.ylabel("Closing Price")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(14, 6))
plt.plot(data.index, data["Volume"], label="Trading Volume")

plt.title("Reliance Industries Trading Volume")
plt.xlabel("Date")
plt.ylabel("Volume")
plt.legend()
plt.grid(True)
plt.show()

## 5. Feature Engineering

In [ ]:
# 20-day moving average
data["MA20"] = data["Close"].rolling(window=20).mean()

# Target: next trading day's closing price
data["Next_Close"] = data["Close"].shift(-1)

# Remove rows created with missing values by rolling/shift operations.
data = data.dropna()

print("Dataset shape after preprocessing:", data.shape)
print("\nRemaining missing values:")
print(data.isnull().sum())

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(data.index, data["Close"], label="Closing Price")
plt.plot(data.index, data["MA20"], label="20-Day Moving Average")

plt.title("Reliance Industries: Closing Price vs 20-Day Moving Average")
plt.xlabel("Date")
plt.ylabel("Price")

plt.legend()
plt.grid(True)
plt.show()

## 6. Select Features and Target

In [ ]:
features = ["Open", "High", "Low", "Close", "Volume", "MA20"]

X = data[features]
y = data["Next_Close"]

print("Features:", features)
print("X shape:", X.shape)
print("y shape:", y.shape)

## 7. Train-Test Split

In [ ]:
# Keep the time order because this is time-series data.
train_size = int(len(data) * 0.8)

X_train = X.iloc[:train_size]
X_test = X.iloc[train_size:]

y_train = y.iloc[:train_size]
y_test = y.iloc[train_size:]

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

## 8. Model 1 — Linear Regression

In [ ]:
model_lr = LinearRegression()

model_lr.fit(X_train, y_train)

y_pred_lr = model_lr.predict(X_test)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mean_squared_error(y_test, y_pred_lr))
r2_lr = r2_score(y_test, y_pred_lr)

print("Linear Regression Results")
print("-------------------------")
print("MAE :", mae_lr)
print("RMSE:", rmse_lr)
print("R²  :", r2_lr)

In [ ]:
comparison_lr = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred_lr
})

comparison_lr.head(10)

## 9. Model 2 — Random Forest Regression

In [ ]:
model_rf = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)

print("Random Forest Results")
print("---------------------")
print("MAE :", mae_rf)
print("RMSE:", rmse_rf)
print("R²  :", r2_rf)

## 10. Compare the Models

In [ ]:
results = pd.DataFrame({
    "Model": ["Linear Regression", "Random Forest"],
    "MAE": [mae_lr, mae_rf],
    "RMSE": [rmse_lr, rmse_rf],
    "R2": [r2_lr, r2_rf]
})

results

In [ ]:
# Lower RMSE means smaller prediction error.
best_model_name = results.loc[results["RMSE"].idxmin(), "Model"]

if best_model_name == "Linear Regression":
    best_model = model_lr
    best_predictions = y_pred_lr
    best_mae = mae_lr
    best_rmse = rmse_lr
    best_r2 = r2_lr
else:
    best_model = model_rf
    best_predictions = y_pred_rf
    best_mae = mae_rf
    best_rmse = rmse_rf
    best_r2 = r2_rf

print("Selected model:", best_model_name)

## 11. Actual vs Predicted Price

In [ ]:
plt.figure(figsize=(14, 6))

plt.plot(
    y_test.index,
    y_test.values,
    label="Actual Price"
)

plt.plot(
    y_test.index,
    best_predictions,
    label=f"{best_model_name} Prediction"
)

plt.title("Actual vs Predicted Reliance Stock Price")
plt.xlabel("Date")
plt.ylabel("Closing Price")

plt.legend()
plt.grid(True)
plt.show()

## 12. Final Model Performance

In [ ]:
print("FINAL MODEL PERFORMANCE")
print("-----------------------")
print("Model :", best_model_name)
print("MAE   :", best_mae)
print("RMSE  :", best_rmse)
print("R²    :", best_r2)

## 13. Save the Best Model

In [ ]:
joblib.dump(best_model, "best_stock_model.pkl")
print("Best model saved as best_stock_model.pkl")

## 14. Prediction Function

In [ ]:
def predict_stock_price(open_price, high_price, low_price, close_price, volume, ma20):
    input_data = pd.DataFrame({
        "Open": [open_price],
        "High": [high_price],
        "Low": [low_price],
        "Close": [close_price],
        "Volume": [volume],
        "MA20": [ma20]
    })

    prediction = best_model.predict(input_data)

    return prediction[0]

## 15. Test a Prediction Using the Latest Available Data

In [ ]:
latest_data = data.iloc[-1]

prediction = predict_stock_price(
    latest_data["Open"],
    latest_data["High"],
    latest_data["Low"],
    latest_data["Close"],
    latest_data["Volume"],
    latest_data["MA20"]
)

print("======================================")
print("   RELIANCE STOCK PRICE PREDICTION")
print("======================================")
print("Selected Model:", best_model_name)
print("Latest Closing Price:", round(float(latest_data["Close"]), 2))
print("Predicted Next-Day Closing Price:", round(float(prediction), 2))
print("======================================")

## 16. Gradio Prediction Interface

In [ ]:
import gradio as gr

def predict_from_interface(open_price, high_price, low_price, close_price, volume, ma20):
    prediction = predict_stock_price(
        open_price,
        high_price,
        low_price,
        close_price,
        volume,
        ma20
    )
    return f"₹{float(prediction):.2f}"

interface = gr.Interface(
    fn=predict_from_interface,
    inputs=[
        gr.Number(label="Open Price", value=float(latest_data["Open"])),
        gr.Number(label="High Price", value=float(latest_data["High"])),
        gr.Number(label="Low Price", value=float(latest_data["Low"])),
        gr.Number(label="Close Price", value=float(latest_data["Close"])),
        gr.Number(label="Trading Volume", value=float(latest_data["Volume"])),
        gr.Number(label="20-Day Moving Average", value=float(latest_data["MA20"]))
    ],
    outputs=gr.Textbox(label="Predicted Next-Day Closing Price"),
    title="Reliance Stock Price Prediction",
    description="Estimate the next trading day's closing price using the trained machine learning model."
)

interface.launch(share=False)

## Project Note

The model estimates the next trading day's closing price from historical market features. It is a machine-learning estimation and should not be presented as a guaranteed future stock price.